# Gravitino Spark Lineage support

In [6]:
import pyspark
import os
import sys
from pyspark.sql import SparkSession

spark_home = "/home/jovyan/spark-3.4.2-bin-hadoop3"
gravitino_connector_jar = os.getenv('SPARK_CONNECTOR_JAR')
os.environ['HADOOP_USER_NAME']="anonymous"

spark = SparkSession.builder \
    .appName("Gravitino Spark Lineage example") \
    .config("spark.plugins", "org.apache.gravitino.spark.connector.plugin.GravitinoSparkPlugin") \
    .config("spark.jars", f"/tmp/gravitino/spark/packages/iceberg-spark-runtime-3.4_2.12-1.6.1.jar,/tmp/gravitino/spark/packages/{gravitino_connector_jar},/tmp/gravitino/spark/packages/openlineage-spark_2.12-1.31.0-datastrato-1.jar") \
    .config("spark.sql.gravitino.uri", "http://gravitino:8090") \
    .config("spark.sql.gravitino.metalake", "metalake_demo") \
    .config("spark.sql.gravitino.enableIcebergSupport", "true") \
    .config("spark.locality.wait.node", "0") \
    .config("spark.sql.warehouse.dir", "hdfs://hive:9000/user/hive/warehouse") \
    .config("spark.extraListeners", "io.openlineage.spark.agent.OpenLineageSparkListener") \
    .config("spark.openlineage.transport.type", "http") \
    .config("spark.openlineage.transport.url", "http://gravitino:8090") \
    .config("spark.openlineage.transport.endpoint", "/api/lineage") \
    .config("spark.openlineage.namespace", "metalake_demo") \
    .config("spark.openlineage.appName", "openlineage_spark_job") \
    .config("spark.openlineage.columnLineage.datasetLineageEnabled", "true") \
    .enableHiveSupport() \
    .getOrCreate()

In [7]:
spark.sql("use catalog_hive")
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|  default|
|  product|
|    sales|
+---------+



In [4]:
spark.sql("CREATE DATABASE IF NOT EXISTS product;")
spark.sql("USE product;")
spark.sql("CREATE TABLE IF NOT EXISTS employees (id INT, name STRING, age INT) PARTITIONED BY (department STRING) STORED AS PARQUET;")
spark.sql("DESC TABLE EXTENDED employees;").show()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|                  id|                 int|   null|
|                name|              string|   null|
|                 age|                 int|   null|
|          department|              string|   null|
|# Partition Infor...|                    |       |
|          # col_name|           data_type|comment|
|          department|              string|   null|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|                Name|   product.employees|       |
|                Type|             MANAGED|       |
|            Location|hdfs://1681fef845...|       |
|               Owner|           anonymous|       |
|    Table Properties|[hive.stored-as=P...|       |
+--------------------+--------------------+-------+



In [5]:
spark.sql("INSERT OVERWRITE TABLE employees PARTITION(department='Engineering') VALUES (1, 'John Doe', 30), (2, 'Jane Smith', 28);")
spark.sql("INSERT OVERWRITE TABLE employees PARTITION(department='Marketing') VALUES (3, 'Mike Brown', 32);")
spark.sql("SELECT * from employees").show()

+---+----------+---+-----------+
| id|      name|age| department|
+---+----------+---+-----------+
|  2|Jane Smith| 28|Engineering|
|  1|  John Doe| 30|Engineering|
|  3|Mike Brown| 32|  Marketing|
+---+----------+---+-----------+

